In [ ]:
!pip install torch torchvision timm huggingface_hub Pillow qiskit qiskit-aer

In [6]:
from huggingface_hub import hf_hub_download
import torch, timm
from PIL import Image
from torchvision import transforms

# Download weights
vit_path = hf_hub_download(
    repo_id="canada-guesser/canadian_streetview_cities_models",
    filename="vit_model/swinv2_base_window12_192_0_finetuned_canadian_streetview.bin"
)

# Load model
model = timm.create_model("swinv2_base_window12_192", pretrained=False, num_classes=15)
model.load_state_dict(torch.load(vit_path, map_location="cpu"))
model.eval()

# Run inference
transform = transforms.Compose([
    transforms.Resize((192, 192)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

class_names = [
    "Calgary", "Charlottetown", "Edmonton", "Halifax", "Hamilton",
    "Kitchener-Waterloo", "Montreal", "Ottawa-Gatineau", "Quebec City", "Saskatoon",
    "St Johns", "Toronto", "Vancouver", "Victoria", "Winnipeg",
]

img = Image.open("C:\\_projects\\qiskit-examples\\image.png").convert("RGB")
x = transform(img).unsqueeze(0)

with torch.no_grad():
    pred = model(x)

print("City:", class_names[pred.argmax().item()])

City: Vancouver


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import copy
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


class QuantumPruner:
    """
    Quantum-assisted neural network pruner using amplitude encoding.

    For each weight chunk of size n_qubits the following circuit is built:

      Step 1  Ry amplitude encoding
              θᵢ = 2·arcsin(√wᵢ_norm)  →  P(|1⟩ᵢ) = wᵢ_norm
              This directly maps weight magnitude to qubit excitation probability.

      Step 2  Nearest-neighbour CX entanglement
              Correlates adjacent qubits so that the importance of weight i
              is influenced by its neighbour — something classical L1/L2
              magnitude scoring cannot express.

      Step 3  One Grover diffusion step   2|ψ⟩⟨ψ| − I
              Reflects the current state about itself, amplifying the gap
              between dominant (high-weight) and weak basis states.

    Measured per-qubit probabilities Q form a quantum bias term:
        importance_i = |w_i| · (1 + λ · Q_i)
    Weights below the resulting percentile threshold are zeroed.

    Large layers are handled via stratified sampling: weight magnitudes are
    sorted and max_circuits representative chunks are drawn uniformly across
    the magnitude range; scores are then interpolated back to every weight.
    """

    def __init__(
        self,
        n_qubits: int = 6,
        shots: int = 512,
        max_circuits_per_layer: int = 32,
        seed: int = 42,
    ):
        self.n_qubits = n_qubits
        self.shots = shots
        self.max_circuits = max_circuits_per_layer
        self.rng = np.random.default_rng(seed)
        self.sim = AerSimulator()

    # ── Circuit construction ──────────────────────────────────────────────────

    def build_circuit(self, chunk_norm: np.ndarray) -> QuantumCircuit:
        """
        Build the amplitude-encoding + entanglement + Grover-diffusion circuit.

        chunk_norm : 1-D array of length n_qubits, values in [0, 1]
        """
        n = len(chunk_norm)
        qc = QuantumCircuit(n)

        # Step 1: Ry amplitude encoding
        for i, w in enumerate(chunk_norm):
            theta = 2.0 * np.arcsin(np.sqrt(float(np.clip(w, 1e-9, 1.0 - 1e-9))))
            qc.ry(theta, i)

        # Step 2: Linear CX entanglement
        for i in range(n - 1):
            qc.cx(i, i + 1)

        # Step 3: Grover diffusion  D = H X MCX X H  (reflects about |ψ⟩)
        if n >= 2:
            qc.h(range(n))
            qc.x(range(n))
            qc.h(n - 1)
            qc.mcx(list(range(n - 1)), n - 1)   # multi-controlled X on last qubit
            qc.h(n - 1)
            qc.x(range(n))
            qc.h(range(n))

        qc.measure_all()
        return qc

    def _run_circuit(self, chunk_norm: np.ndarray) -> np.ndarray:
        """Simulate circuit; return per-qubit P(|1⟩) probabilities."""
        n = len(chunk_norm)
        qc = self.build_circuit(chunk_norm)
        compiled = transpile(qc, self.sim, optimization_level=1)
        counts = self.sim.run(compiled, shots=self.shots).result().get_counts()
        probs = np.zeros(n)
        for bitstring, cnt in counts.items():
            for i, bit in enumerate(reversed(bitstring)):
                if i < n:
                    probs[i] += int(bit) * cnt
        return probs / self.shots

    # ── Importance scoring ────────────────────────────────────────────────────

    def quantum_scores(self, flat_abs: np.ndarray) -> np.ndarray:
        """
        Compute quantum importance scores for a flattened weight tensor.

        - Small layers  (n_chunks ≤ max_circuits): every chunk is evaluated.
        - Large layers: stratified sampling across the magnitude-sorted order,
          followed by linear interpolation to all weight positions.
        """
        n = len(flat_abs)
        w_max = flat_abs.max()
        if w_max == 0.0:
            return np.zeros(n)
        w_norm = flat_abs / w_max
        step = self.n_qubits
        n_chunks = (n + step - 1) // step

        if n_chunks <= self.max_circuits:
            # Full processing
            scores = np.empty(n)
            for s in range(0, n, step):
                e = min(s + step, n)
                scores[s:e] = self._run_circuit(w_norm[s:e])
        else:
            # Stratified sampling across magnitude quantiles
            sorted_idx = np.argsort(w_norm)
            bucket = max(1, n // self.max_circuits)
            sample_means, sample_ranks = [], []
            for b in range(self.max_circuits):
                s = b * bucket
                e = min(s + step, n)
                if e > n:
                    break
                q = self._run_circuit(w_norm[sorted_idx[s:e]])
                sample_means.append(np.mean(q))
                sample_ranks.append(s + (e - s) / 2.0)

            # Interpolate quantum scores back to all rank positions
            all_ranks = np.arange(n, dtype=float)
            q_by_rank = np.interp(all_ranks, sample_ranks, sample_means)
            scores = np.empty(n)
            scores[sorted_idx] = q_by_rank

        return scores

    # ── Pruning API ───────────────────────────────────────────────────────────

    def prune_tensor(
        self,
        weight: torch.Tensor,
        sparsity: float,
        quantum_bias: float = 0.25,
    ):
        """
        Zero the bottom `sparsity` fraction of a weight tensor.

            importance_i = |w_i| · (1 + quantum_bias · Q_i)

        Returns (pruned_weight, bool_mask).
        """
        flat = weight.detach().cpu().numpy().ravel()
        abs_w = np.abs(flat)
        q = self.quantum_scores(abs_w)
        importance = abs_w * (1.0 + quantum_bias * q)
        threshold = np.percentile(importance, sparsity * 100.0)
        mask = (importance >= threshold).reshape(weight.shape)
        pruned_w = torch.tensor(
            weight.detach().cpu().numpy() * mask, dtype=weight.dtype
        )
        return pruned_w, mask

    def apply(
        self,
        model: nn.Module,
        sparsity_config: dict,
        quantum_bias: float = 0.25,
        verbose: bool = True,
    ):
        """
        Prune every nn.Linear layer in *model* using quantum-biased importance.

        sparsity_config keys are substrings matched against module names;
        'default' is the fallback.  Example for SwinV2:
            {'attn': 0.15, 'mlp': 0.40, 'head': 0.10, 'default': 0.30}

        Returns (pruned_model, stats_dict).
        """
        model = copy.deepcopy(model)
        total = pruned_count = 0
        layer_stats = []

        def resolve(name: str) -> float:
            for k, v in sparsity_config.items():
                if k != 'default' and k in name:
                    return v
            return sparsity_config.get('default', 0.30)

        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear):
                continue
            sp = resolve(name)
            new_w, mask = self.prune_tensor(mod.weight, sp, quantum_bias)
            mod.weight = nn.Parameter(new_w)
            n_pruned = int((~mask).sum())
            n_total  = int(mask.size)
            pruned_count += n_pruned
            total        += n_total
            layer_stats.append(dict(
                name=name, shape=list(mod.weight.shape),
                target=sp, actual=n_pruned / n_total,
            ))
            if verbose:
                print(f"  [{sp:.0%}→{n_pruned/n_total:.1%}]  {name}  {list(mod.weight.shape)}")

        overall = pruned_count / total if total else 0.0
        if verbose:
            print(f"\n  Overall sparsity: {overall:.1%}  "
                  f"({pruned_count:,} / {total:,} weights zeroed)")

        return model, dict(
            total=total, pruned=pruned_count,
            overall_sparsity=overall, layers=layer_stats,
        )


In [ ]:
import time
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ── Sparsity schedule ─────────────────────────────────────────────────────────
# SwinV2 layer names: layers.X.blocks.Y.{attn,mlp}.*  and  head.*
# Attention projections are information bottlenecks → prune conservatively.
# MLP feed-forward blocks carry more redundancy → prune aggressively.
SPARSITY_CONFIG = {
    'attn':    0.15,   # attention Q/K/V + projection
    'mlp':     0.40,   # feed-forward fc1 / fc2
    'head':    0.10,   # final classifier head
    'default': 0.30,   # everything else
}

pruner = QuantumPruner(n_qubits=6, shots=512, max_circuits_per_layer=32)

print("Applying quantum-assisted pruning …\n")
t0 = time.time()
model_pruned, stats = pruner.apply(model, SPARSITY_CONFIG, quantum_bias=0.25)
print(f"\nCompleted in {time.time() - t0:.1f}s")

# ── Inference comparison ──────────────────────────────────────────────────────
model_pruned.eval()
with torch.no_grad():
    logits_orig   = model(x)
    logits_pruned = model_pruned(x)

probs_orig   = F.softmax(logits_orig,   dim=1)[0].numpy()
probs_pruned = F.softmax(logits_pruned, dim=1)[0].numpy()
top3_orig    = np.argsort(probs_orig)[::-1][:3]
top3_pruned  = np.argsort(probs_pruned)[::-1][:3]

print("\n┌────────────────────────────────────────────────────────────┐")
print("│                   Inference Comparison                     │")
print("├──────────────────────────────┬─────────────────────────────┤")
print("│  Original                    │  Pruned                     │")
print("├──────────────────────────────┼─────────────────────────────┤")
for rank, (io, ip) in enumerate(zip(top3_orig, top3_pruned), 1):
    print(f"│  {rank}. {class_names[io]:<24s} {probs_orig[io]:5.1%}  │"
          f"  {rank}. {class_names[ip]:<24s} {probs_pruned[ip]:5.1%}  │")
print("└──────────────────────────────┴─────────────────────────────┘")

total_all  = sum(p.numel() for p in model.parameters())
pruned_all = sum((p == 0).sum().item() for p in model_pruned.parameters())
print(f"\n  Total parameters  : {total_all:,}")
print(f"  Zeroed parameters : {pruned_all:,}  ({pruned_all/total_all:.1%} of all params)")
print(f"  Linear sparsity   : {stats['overall_sparsity']:.1%}")

# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Quantum-Assisted Pruning  —  SwinV2-Base", fontsize=13, fontweight='bold')

groups  = {'attn': [], 'mlp': [], 'head': [], 'other': []}
colors  = {'attn': '#4C72B0', 'mlp': '#DD8452', 'head': '#55A868', 'other': '#C44E52'}
targets = {'attn': 0.15,      'mlp': 0.40,      'head': 0.10,      'other': 0.30}
for ls in stats['layers']:
    key = next((k for k in ('attn', 'mlp', 'head') if k in ls['name']), 'other')
    groups[key].append(ls['actual'])

ax0  = axes[0]
keys = [g for g in groups if groups[g]]
for i, grp in enumerate(keys):
    vals = groups[grp]
    ax0.bar(i, np.mean(vals), color=colors[grp], alpha=0.82, label=grp,
            yerr=(np.std(vals) if len(vals) > 1 else 0), capsize=5)
    ax0.scatter([i] * len(vals), vals, color=colors[grp], alpha=0.35, s=18, zorder=3)
ax0.axhline(stats['overall_sparsity'], ls='--', color='black', lw=1.3,
            label=f"overall ({stats['overall_sparsity']:.1%})")
ax0.set_xticks(range(len(keys)))
ax0.set_xticklabels(keys, fontsize=11)
ax0.set_ylabel("Actual sparsity")
ax0.set_title("Sparsity by layer type")
ax0.set_ylim(0, 0.55)
ax0.legend(fontsize=9)

ax1   = axes[1]
x_pos = np.arange(len(class_names))
bar_w = 0.35
ax1.bar(x_pos - bar_w/2, probs_orig,   width=bar_w, label='Original', color='#4C72B0', alpha=0.82)
ax1.bar(x_pos + bar_w/2, probs_pruned, width=bar_w, label='Pruned',   color='#DD8452', alpha=0.82)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel("Softmax probability")
ax1.set_title("Class probabilities: Original vs Pruned")
ax1.legend()
plt.tight_layout()
plt.savefig("pruning_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Quantum circuit introspection ─────────────────────────────────────────────
example_weights = np.array([0.85, 0.12, 0.67, 0.34, 0.91, 0.05])
w_norm = example_weights / example_weights.max()
qc_demo = pruner.build_circuit(w_norm)
print("\nQuantum importance circuit for a 6-weight chunk")
print("=" * 65)
print(qc_demo.draw(output='text', fold=120))
print(f"  Depth : {qc_demo.depth()}   Gates : {dict(qc_demo.count_ops())}")
measured = pruner._run_circuit(w_norm)
print(f"  |w| normalised       : {np.round(w_norm, 3)}")
print(f"  P(|1⟩) — Ry only    : {np.round(w_norm, 3)}")
print(f"  P(|1⟩) — full circuit: {np.round(measured, 3)}")
print("\n  Grover diffusion sharpens contrast: high-magnitude qubits gain probability,")
print("  low-magnitude qubits lose it — making the pruning threshold more decisive.")
